# Deformation-grid visualization across methods

For each method that produced a full-DVF run (per-slice `.npy` in
`results_full_dvf/{method}/slice_z{Z:03d}.npy`), this notebook draws
the warped grid before and after correction on a few representative
slices:

* **Wall slice** `z=12` — the dense fold core that defeats classical SLSQP.
* **Mid-density** `z=100` — typical clean slice the manuscript run handles easily.
* **Tail slice** `z=300` — late in the volume, moderate folding.
* **Easy slice** `z=527` — last slice, minimal folding.

Layout: rows = methods, columns = [input grid, corrected grid].
A fold appears as grid lines crossing themselves (the triangle area
goes negative). The title shows the residual fold count and the L1/L2
distance from the input.

Methods compared:
* manuscript SLSQP (cluster windowed run that produced `output/2d_real_full/slices/`)
* slsqp_windowed (fresh harness reimplementation)
* m10 harmonic_l2_polished — always-feasibility seed
* m14 l2_refine_repair — L2-refined m10
* m14_l1 l1_refine_repair — L1-anchor refined m10 (current winner)

In [ ]:
import os, sys
sys.path.insert(0, '.')
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import harness as H
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d

vol = np.load(H.DATA_PATH, mmap_mode='r')
D, Hh, Ww = vol.shape[1], vol.shape[2], vol.shape[3]
print(f'volume: D={D}  H={Hh}  W={Ww}')

# (method NAME, per-slice .npy directory). The manuscript run lives in a
# different folder so it gets a special-case path resolver below.
METHODS = [
    ('manuscript SLSQP (cluster)',  None,                                'manuscript_slsqp'),
    ('slsqp_windowed (fresh)',      'results_full_dvf/slsqp_windowed',   'slsqp_windowed'),
    ('m10 harmonic_l2_polished',    'results_full_dvf/harmonic_l2_polished', 'harmonic_l2_polished'),
    ('m14 l2_refine_repair',        'results_full_dvf/l2_refine_repair', 'l2_refine_repair'),
    ('m14_l1 l1_refine_repair',     'results_full_dvf/l1_refine_repair', 'l1_refine_repair'),
]

MANUSCRIPT_DIR = os.path.abspath(
    os.path.join('..', '..', 'manuscript', 'output', '2d_real_full', 'slices'))

TARGET_ZS = [12, 100, 300, 527]

In [ ]:
def load_phi_out(method_dir, csv_method_name, z):
    """Return (2, H, W) corrected slice or None if missing.

    Manuscript SLSQP slices live in ../../manuscript/output/2d_real_full/slices/
    as (3, 1, H, W). The wall-breaker methods save (2, H, W) directly.
    """
    if csv_method_name == 'manuscript_slsqp':
        fn = os.path.join(MANUSCRIPT_DIR, f'slice_z{z:03d}.npy')
        if not os.path.isfile(fn):
            return None
        out = np.load(fn)
        if out.ndim == 4 and out.shape[0] == 3:
            return np.stack([out[1, 0], out[2, 0]])
        if out.ndim == 3 and out.shape[0] == 3:
            return np.stack([out[1], out[2]])
        return out
    if method_dir is None:
        return None
    fn = os.path.join(method_dir, f'slice_z{z:03d}.npy')
    if not os.path.isfile(fn):
        return None
    return np.load(fn)


def draw_warped_grid(ax, phi_dy_dx, *, stride=8, colors='k', linewidths=0.3,
                     title=''):
    """Draw the deformed grid as a wireframe via LineCollection.
    ``stride`` controls how many grid lines we draw; stride=1 is every line,
    stride=8 draws every 8th (much faster for 320x456)."""
    H_, W_ = phi_dy_dx.shape[1], phi_dy_dx.shape[2]
    yy, xx = np.mgrid[0:H_, 0:W_].astype(float)
    def_x = xx + phi_dy_dx[1]
    def_y = yy + phi_dy_dx[0]
    rows = [np.column_stack([def_x[i, :], def_y[i, :]]) for i in range(0, H_, stride)]
    cols = [np.column_stack([def_x[:, j], def_y[:, j]]) for j in range(0, W_, stride)]
    ax.add_collection(LineCollection(rows + cols, colors=colors,
                                      linewidths=linewidths))
    pad = max(W_, H_) * 0.03
    ax.set_xlim(def_x.min() - pad, def_x.max() + pad)
    # image-style Y axis: row 0 at top, increasing downward.
    ax.set_ylim(def_y.max() + pad, def_y.min() - pad)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9)


def metric_strip(phi_in, phi_out):
    """Short metric summary for a panel title."""
    m = H.metrics(phi_out)
    L2 = H.l2_delta(phi_out, phi_in)
    L1 = H.l1_delta(phi_out, phi_in)
    return (f'n_neg={m["tri_neg"]:4d}  min_T={m["tri_min"]:+.4f}  '
            f'L2={L2:.1f}  L1={L1:.0f}')

In [ ]:
# Build one large figure per target z: rows = methods, columns = [in, out].
# Stride controls density of drawn grid lines. 320x456 is too dense to draw
# every line in a static figure; stride=8 keeps the file size reasonable.
#
# Layout: set_aspect('equal') locks each panel to the data's W/H ratio,
# so a 13in figure leaves whitespace beside it. Size the figure to the
# data aspect ratio exactly and tighten wspace.
STRIDE = 8

for z in TARGET_ZS:
    phi_in = np.stack([vol[1, z].copy(), vol[2, z].copy()])
    init_m = H.metrics(phi_in)

    panel_w = 4.5
    panel_h = panel_w * phi_in.shape[1] / phi_in.shape[2]    # match W/H ratio
    fig, axes = plt.subplots(len(METHODS), 2,
                              figsize=(2 * panel_w + 0.3,
                                       len(METHODS) * panel_h + 0.7),
                              gridspec_kw={'wspace': 0.02, 'hspace': 0.25})
    for row_idx, (label, method_dir, csv_name) in enumerate(METHODS):
        phi_out = load_phi_out(method_dir, csv_name, z)
        ax_in, ax_out = axes[row_idx]
        draw_warped_grid(
            ax_in, phi_in, stride=STRIDE,
            title=f'{label}\nINPUT  n_neg={init_m["tri_neg"]}  min_T={init_m["tri_min"]:+.3f}')
        if phi_out is None:
            ax_out.text(0.5, 0.5, f'(no output for {csv_name})',
                        ha='center', va='center', fontsize=10,
                        transform=ax_out.transAxes)
            ax_out.set_xticks([]); ax_out.set_yticks([])
            ax_out.set_title('—', fontsize=9)
        else:
            draw_warped_grid(
                ax_out, phi_out, stride=STRIDE,
                title=f'OUTPUT  {metric_strip(phi_in, phi_out)}')
    fig.suptitle(f'z = {z}   (warped grid; stride={STRIDE})',
                  fontsize=13, fontweight='bold')
    fig.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.02)
    plt.show()

## Zoom on the wall slice (z = 12)

On the wall slice the relevant differences live inside a small fold-core region
(roughly y ∈ [120, 220], x ∈ [140, 240] in pixel coordinates). The next cell
draws every grid line (stride = 1) but restricts the view to that bounding
box so the per-method behaviour is visible.

In [ ]:
Z_ZOOM = 12
BOX_Y = (120, 220)
BOX_X = (140, 240)

phi_in = np.stack([vol[1, Z_ZOOM].copy(), vol[2, Z_ZOOM].copy()])
y0, y1 = BOX_Y; x0, x1 = BOX_X

def draw_zoom(ax, phi_dy_dx, title, stride=1):
    H_, W_ = phi_dy_dx.shape[1], phi_dy_dx.shape[2]
    yy, xx = np.mgrid[0:H_, 0:W_].astype(float)
    def_x = xx + phi_dy_dx[1]
    def_y = yy + phi_dy_dx[0]
    rows = [np.column_stack([def_x[i, x0:x1], def_y[i, x0:x1]])
            for i in range(y0, y1, stride)]
    cols = [np.column_stack([def_x[y0:y1, j], def_y[y0:y1, j]])
            for j in range(x0, x1, stride)]
    ax.add_collection(LineCollection(rows + cols, colors='k',
                                      linewidths=0.5))
    all_x = np.concatenate([r[:, 0] for r in rows + cols])
    all_y = np.concatenate([r[:, 1] for r in rows + cols])
    pad = max(BOX_X[1] - BOX_X[0], BOX_Y[1] - BOX_Y[0]) * 0.05
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.max() + pad, all_y.min() - pad)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9)

# Same aspect-aware figsize for the zoom (BOX_X width vs BOX_Y height).
panel_w = 4.0
panel_h = panel_w * (BOX_Y[1] - BOX_Y[0]) / (BOX_X[1] - BOX_X[0])
fig, axes = plt.subplots(len(METHODS), 2,
                          figsize=(2 * panel_w + 0.3,
                                   len(METHODS) * panel_h + 0.8),
                          gridspec_kw={'wspace': 0.02, 'hspace': 0.3})
init_m = H.metrics(phi_in)
for row_idx, (label, method_dir, csv_name) in enumerate(METHODS):
    phi_out = load_phi_out(method_dir, csv_name, Z_ZOOM)
    ax_in, ax_out = axes[row_idx]
    draw_zoom(
        ax_in, phi_in,
        title=f'{label}\nINPUT (zoom)  n_neg={init_m["tri_neg"]}')
    if phi_out is None:
        ax_out.text(0.5, 0.5, '(no output)', ha='center', va='center',
                    transform=ax_out.transAxes)
        ax_out.set_xticks([]); ax_out.set_yticks([])
    else:
        draw_zoom(
            ax_out, phi_out,
            title=f'OUTPUT (zoom)  {metric_strip(phi_in, phi_out)}')
fig.suptitle(f'z = {Z_ZOOM} fold-core zoom (every grid line)  '
             f'box y={BOX_Y}, x={BOX_X}', fontsize=12, fontweight='bold')
fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.02)
plt.show()

## Per-cell `min(T1, T2)` heatmap comparison

Same slices, but instead of the wireframe grid we draw the per-cell feasibility
scalar `min(T1, T2)` as a heatmap. Red ≤ 0 = folded; white ≈ 0 = on the boundary;
blue > 0 = safely feasible. The cyan contour outlines folded cells.

This makes the feasibility difference between methods very visible — even
when the wireframe looks similar, the heatmap shows which cells are still
near the wall.

In [ ]:
def draw_T_min_heatmap(ax, phi_dy_dx, vmax, title):
    T1, T2 = _triangle_areas_2d(phi_dy_dx[0], phi_dy_dx[1])
    Tmin = np.minimum(T1, T2)
    im = ax.imshow(Tmin, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.contour((Tmin <= 0).astype(float), levels=[0.5],
                colors='cyan', linewidths=0.4)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9)
    return im

for z in TARGET_ZS:
    phi_in = np.stack([vol[1, z].copy(), vol[2, z].copy()])
    init_m = H.metrics(phi_in)
    vmax = max(abs(init_m['tri_min']), init_m['tri_min'], 0.1)

    # imshow with default aspect uses pixel aspect = 1 (W:H = 456:320 ~= 1.425).
    panel_w = 4.5
    panel_h = panel_w * 320 / 456
    fig, axes = plt.subplots(len(METHODS), 2,
                              figsize=(2 * panel_w + 0.8,    # +space for colorbar
                                       len(METHODS) * panel_h + 0.7),
                              gridspec_kw={'wspace': 0.02, 'hspace': 0.25})
    last_im = None
    for row_idx, (label, method_dir, csv_name) in enumerate(METHODS):
        phi_out = load_phi_out(method_dir, csv_name, z)
        ax_in, ax_out = axes[row_idx]
        last_im = draw_T_min_heatmap(
            ax_in, phi_in, vmax,
            f'{label}\nINPUT  n_neg={init_m["tri_neg"]}  min_T={init_m["tri_min"]:+.3f}')
        if phi_out is None:
            ax_out.text(0.5, 0.5, '(no output)', ha='center', va='center',
                        transform=ax_out.transAxes)
            ax_out.set_xticks([]); ax_out.set_yticks([])
        else:
            draw_T_min_heatmap(
                ax_out, phi_out, vmax,
                f'OUTPUT  {metric_strip(phi_in, phi_out)}')
    fig.suptitle(f'z = {z}   min(T1, T2) heatmap   (cyan = folded contour)',
                  fontsize=12, fontweight='bold')
    fig.subplots_adjust(left=0.01, right=0.92, top=0.93, bottom=0.02)
    cbar_ax = fig.add_axes([0.93, 0.05, 0.015, 0.85])
    fig.colorbar(last_im, cax=cbar_ax, label='min(T1, T2)')
    plt.show()

## Reading the figures

* **Input column** — same for every method (just the original `phi_in` slice).
  Folded cells form a tangled patch in the warped grid; the heatmap shows the
  fold core as a red splotch.
* **manuscript SLSQP / slsqp_windowed** — on the easy slices the corrected
  grid is essentially equal to the input (only fold cells move). On the wall
  slice you can see residual fold patches because the cluster-windowed solver
  fails to fully heal the densest cores.
* **m10 harmonic_l2_polished** — the corrected grid is *smooth everywhere*
  including outside the fold cores: m10 replaced the whole core with a
  harmonic completion of the ring, sacrificing a lot of L2 for guaranteed
  feasibility.
* **m14 l2_refine_repair** — visible reduction in how much the field was
  perturbed outside the fold cores (closer to manuscript SLSQP in non-core
  cells) while preserving full feasibility in the cores.
* **m14_l1 l1_refine_repair** — most aggressive at returning non-core
  cells to the input; deviation is concentrated in the few cells that must
  move to keep the cores feasible. Heatmap shows the fold-core contour
  has shrunk to a thin band right at threshold.